# Chấm cross-encoder offline cho luồng issue

Chọn **GPU T4**, upload `issue_ce_offline_bundle.zip`. Notebook chỉ chạy inference `BAAI/bge-reranker-base` gốc trên ~1,09 triệu cặp (mỗi issue × top-100 hybrid production); không train, không dùng nhãn.

Kết quả lưu theo shard trên Google Drive. Nếu runtime bị ngắt: kết nối lại, chạy lại **từ đầu tất cả các ô**, shard đã xong sẽ được bỏ qua.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU, rồi kết nối lại.'
print(torch.cuda.get_device_name(0))

In [ ]:
# Lưu shard trên Drive để chạy tiếp được nếu bị ngắt. Không muốn mount: đặt USE_DRIVE = False.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/issue_ce_offline_output'
else:
    OUT = '/content/issue_ce_offline/output'
print(OUT)

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile
uploaded = files.upload()
archives = [n for n in uploaded if n.lower().endswith('.zip')]
assert len(archives) == 1, f'Upload đúng một ZIP; nhận được: {list(uploaded)}'
run_dir = Path('/content/issue_ce_offline')
shutil.rmtree(run_dir / 'data', ignore_errors=True)
run_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as zf:
    zf.extractall(run_dir)
%cd /content/issue_ce_offline

In [ ]:
!pip -q install 'transformers>=4.44,<5' sentencepiece safetensors
!python score_issue_ce_offline.py --data data --out "$OUT" --batch-size 128 --shard-size 250

In [ ]:
# Chỉ cần ce_scores.npy + manifest.json; bỏ shard cho file nhẹ.
import zipfile
with zipfile.ZipFile('/content/issue_ce_offline_result.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in ('ce_scores.npy', 'manifest.json'):
        zf.write(f'{OUT}/{name}', name)
from google.colab import files
files.download('/content/issue_ce_offline_result.zip')